# 06b · Refinamiento local de XGBoost y regresión logística

Este notebook conserva intacto el HPO general del notebook 06 y resuelve dos cuestiones antes de la validación externa:

- Refina XGBoost alrededor del trial 29, ampliando el límite inferior de árboles y guardando `best_iteration` y tiempo de ajuste.
- Sustituye el HPO logístico costoso por una línea base L2 sencilla: cinco valores de `C`, escalado únicamente numérico y comprobación explícita de convergencia.

Solo utiliza filas etiquetadas con `dataset_split == 'train'`. Octubre, noviembre y diciembre de 2022 no se leen ni se usan.

## Dependencias, rutas y configuración

Los resultados se escriben en una carpeta nueva para no sobrescribir el HPO original. Si existen las dos copias de `Datos modelado`, se prioriza la que contiene los resultados del notebook 06 ejecutado.

In [ ]:
# %pip install scikit-learn xgboost optuna matplotlib

from pathlib import Path
import gc
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import sklearn
from IPython.display import display
from optuna.samplers import NSGAIISampler
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Localizamos la raíz del proyecto con independencia del directorio desde el que se abre Jupyter.
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR

# Priorizamos la copia que contiene los resultados reales del HPO ejecutado.
data_candidates = [
    PROJECT_ROOT / 'notebooks' / 'Datos modelado',
    PROJECT_ROOT / 'Datos modelado',
]
MODEL_DATA_DIR = next(
    (path for path in data_candidates if (path / 'hpo_temporal' / 'mejores_hiperparametros.csv').exists()),
    None,
)
if MODEL_DATA_DIR is None:
    raise FileNotFoundError('No se encontró la carpeta hpo_temporal generada por el notebook 06.')

FEATURES_DIR = MODEL_DATA_DIR / 'estacion_hora_features'
HPO_ORIGINAL_DIR = MODEL_DATA_DIR / 'hpo_temporal'
OUTPUT_DIR = MODEL_DATA_DIR / 'hpo_refinamiento'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'risk_class_1h'
TIME_COLUMN = 'fecha_hora_local'
RANDOM_STATE = 42
SEEDS_STABILITY = [13, 42, 97]
HORIZON_HOURS = 1
CHUNK_SIZE = 100_000
MAX_TRAIN_ROWS_PER_FOLD = 250_000
MAX_VALIDATION_ROWS_PER_FOLD = 100_000
N_TRIALS_XGBOOST = 30
# XGBoost ya terminó su refinamiento. Déjalo en False para ejecutar solo la línea base logística.
RUN_XGBOOST_REFINEMENT = False
LOGISTIC_C_VALUES = [0.1, 0.3, 1.0, 3.0, 10.0]
LOGISTIC_MAX_ITER = 700
LOGISTIC_TOL = 1e-3
F1_TOLERANCE = 0.003

print('Carpeta de datos utilizada:', MODEL_DATA_DIR)
print('Resultados del refinamiento:', OUTPUT_DIR)
print('Versiones:', {'scikit-learn': sklearn.__version__, 'optuna': optuna.__version__})


## Variables y archivos autorizados

Se reproducen las variables del HPO original. La selección física de archivos admite únicamente 2019, 2021 y enero–septiembre de 2022; 2020 continúa fuera del modelo principal.

In [ ]:
NUMERIC_FEATURES = [
    'capacity', 'bikes_available', 'docks_available', 'reservations_count',
    'occupancy_ratio', 'light', 'weather_available',
    'uv_radiation_median_mw_m2', 'wind_speed_median_m_s',
    'wind_direction_sin_mean', 'wind_direction_cos_mean',
    'temperature_median_c', 'relative_humidity_median_pct',
    'barometric_pressure_median_mb', 'solar_radiation_median_w_m2',
    'precipitation_mean_l_m2', 'precipitation_max_l_m2',
    'n_temperature', 'n_relative_humidity', 'n_precipitation',
    'hour', 'day_of_week', 'month', 'week_of_year',
    'occupancy_ratio_lag_1h', 'occupancy_ratio_lag_2h', 'occupancy_ratio_lag_24h',
    'bikes_available_lag_1h', 'bikes_available_lag_2h', 'bikes_available_lag_24h',
    'net_flow_lag_1h', 'net_flow_lag_2h', 'net_flow_lag_24h',
    'departures_count_lag_1h', 'departures_count_lag_2h', 'departures_count_lag_24h',
    'arrivals_count_lag_1h', 'arrivals_count_lag_2h', 'arrivals_count_lag_24h',
    'occupancy_ratio_mean_previous_3h', 'occupancy_ratio_mean_previous_24h',
    'net_flow_mean_previous_3h', 'net_flow_mean_previous_24h',
    'departures_count_mean_previous_3h', 'departures_count_mean_previous_24h',
    'arrivals_count_mean_previous_3h', 'arrivals_count_mean_previous_24h',
]
CATEGORICAL_FEATURES = ['station_id', 'tipo_dia']
MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
READ_COLUMNS = MODEL_FEATURES + [TARGET, 'dataset_split', TIME_COLUMN]

def file_year_month(file_path: Path) -> tuple[int, int]:
    """Extrae AAAA y MM del nombre estacion_hora_features_AAAAMM.csv."""
    token = file_path.stem.rsplit('_', maxsplit=1)[-1]
    return int(token[:4]), int(token[4:6])

all_feature_files = sorted(FEATURES_DIR.glob('estacion_hora_features_*.csv'))
feature_files = [
    file_path for file_path in all_feature_files
    if file_year_month(file_path)[0] in {2019, 2021}
    or (file_year_month(file_path)[0] == 2022 and file_year_month(file_path)[1] <= 9)
]
if not feature_files:
    raise FileNotFoundError(f'No se encontraron CSV de variables en {FEATURES_DIR}')

# Esta aserción impide que octubre–diciembre entren accidentalmente en el refinamiento.
assert all(not (file_year_month(path)[0] == 2022 and file_year_month(path)[1] >= 10) for path in feature_files)
print(f'Particiones autorizadas: {len(feature_files)}; última: {feature_files[-1].name}')


## Los mismos cuatro folds temporales

Las fronteras coinciden con el notebook 06. Todas las estaciones de una misma hora pertenecen al mismo lado de la división y se elimina la última hora de train para impedir que su objetivo `t+1` alcance la validación.

In [ ]:
FOLD_SPECS = [
    {'fold': 'fold_1', 'validation_start': '2021-01-01 00:00:00', 'validation_end': '2021-04-01 00:00:00'},
    {'fold': 'fold_2', 'validation_start': '2021-07-01 00:00:00', 'validation_end': '2021-10-01 00:00:00'},
    {'fold': 'fold_3', 'validation_start': '2022-01-01 00:00:00', 'validation_end': '2022-04-01 00:00:00'},
    {'fold': 'fold_4', 'validation_start': '2022-07-01 00:00:00', 'validation_end': '2022-10-01 00:00:00'},
]
for spec in FOLD_SPECS:
    spec['validation_start'] = pd.Timestamp(spec['validation_start'])
    spec['validation_end'] = pd.Timestamp(spec['validation_end'])
    spec['train_end_exclusive'] = spec['validation_start'] - pd.Timedelta(hours=HORIZON_HOURS)

def parse_local_hour(values: pd.Series) -> pd.Series:
    """Convierte la hora local sin mezclar offsets de horario de verano."""
    return pd.to_datetime(values.astype('string').str.slice(0, 19), errors='coerce')

def temporal_masks(local_hour: pd.Series, spec: dict) -> tuple[pd.Series, pd.Series]:
    """Crea ventanas disjuntas de train y validación interna."""
    train_mask = local_hour.lt(spec['train_end_exclusive'])
    validation_mask = local_hour.ge(spec['validation_start']) & local_hour.lt(spec['validation_end'])
    return train_mask, validation_mask

display(pd.DataFrame(FOLD_SPECS)[['fold', 'train_end_exclusive', 'validation_start', 'validation_end']])


## Reproducción del muestreo del notebook 06

Se cuentan las clases y se usan las mismas semillas y límites por fold. De esta manera, la comparación entre el HPO original y el refinamiento se realiza sobre las mismas ventanas y muestras.

In [ ]:
fold_class_counts = {
    spec['fold']: {part: {0: 0, 1: 0, 2: 0} for part in ('train', 'validation')}
    for spec in FOLD_SPECS
}

for file_path in feature_files:
    for chunk in pd.read_csv(
        file_path, usecols=[TIME_COLUMN, TARGET, 'dataset_split'],
        chunksize=CHUNK_SIZE, low_memory=False,
    ):
        chunk = chunk.loc[chunk['dataset_split'].eq('train') & chunk[TARGET].notna()].copy()
        if chunk.empty:
            continue
        local_hour = parse_local_hour(chunk[TIME_COLUMN])
        for spec in FOLD_SPECS:
            train_mask, validation_mask = temporal_masks(local_hour, spec)
            for part_name, mask in [('train', train_mask), ('validation', validation_mask)]:
                counts = chunk.loc[mask, TARGET].astype(int).value_counts()
                for class_value, count in counts.items():
                    fold_class_counts[spec['fold']][part_name][int(class_value)] += int(count)

def class_sampling_probabilities(class_counts: dict, maximum_rows: int) -> dict:
    """Muestrea cada clase en la misma proporción que el HPO original."""
    total = sum(class_counts.values())
    if total <= maximum_rows:
        return {class_value: 1.0 for class_value in class_counts}
    return {
        class_value: min(1.0, maximum_rows / total) if count else 0.0
        for class_value, count in class_counts.items()
    }

sampling_probabilities = {}
rng_by_group = {}
for fold_index, spec in enumerate(FOLD_SPECS):
    fold_name = spec['fold']
    for part_index, part_name in enumerate(('train', 'validation')):
        maximum_rows = MAX_TRAIN_ROWS_PER_FOLD if part_name == 'train' else MAX_VALIDATION_ROWS_PER_FOLD
        sampling_probabilities[(fold_name, part_name)] = class_sampling_probabilities(
            fold_class_counts[fold_name][part_name], maximum_rows
        )
        for class_value in (0, 1, 2):
            seed = RANDOM_STATE + 100 * fold_index + 10 * part_index + class_value
            rng_by_group[(fold_name, part_name, class_value)] = np.random.default_rng(seed)

sample_parts = {spec['fold']: {'train': [], 'validation': []} for spec in FOLD_SPECS}
for file_path in feature_files:
    for chunk in pd.read_csv(file_path, usecols=READ_COLUMNS, chunksize=CHUNK_SIZE, low_memory=False):
        chunk = chunk.loc[chunk['dataset_split'].eq('train') & chunk[TARGET].notna()].copy()
        if chunk.empty:
            continue
        chunk['__local_hour'] = parse_local_hour(chunk[TIME_COLUMN])
        chunk[TARGET] = chunk[TARGET].astype('int8')
        for spec in FOLD_SPECS:
            fold_name = spec['fold']
            train_mask, validation_mask = temporal_masks(chunk['__local_hour'], spec)
            for part_name, temporal_mask in [('train', train_mask), ('validation', validation_mask)]:
                candidate = chunk.loc[temporal_mask]
                selected_positions = []
                for class_value in (0, 1, 2):
                    class_positions = np.flatnonzero(candidate[TARGET].to_numpy() == class_value)
                    if len(class_positions) == 0:
                        continue
                    probability = sampling_probabilities[(fold_name, part_name)][class_value]
                    rng = rng_by_group[(fold_name, part_name, class_value)]
                    selected_positions.append(class_positions[rng.random(len(class_positions)) < probability])
                if selected_positions:
                    positions = np.concatenate(selected_positions)
                    if len(positions):
                        sample_parts[fold_name][part_name].append(candidate.iloc[positions].copy())

raw_folds = []
for spec in FOLD_SPECS:
    fold_name = spec['fold']
    train_frame = pd.concat(sample_parts[fold_name]['train'], ignore_index=True)
    validation_frame = pd.concat(sample_parts[fold_name]['validation'], ignore_index=True)
    assert train_frame['dataset_split'].eq('train').all()
    assert validation_frame['dataset_split'].eq('train').all()
    assert train_frame['__local_hour'].max() < validation_frame['__local_hour'].min()
    raw_folds.append({'fold': fold_name, 'train': train_frame, 'validation': validation_frame})
    print(f'{fold_name}: train={len(train_frame):,}; validation={len(validation_frame):,}')

del sample_parts
gc.collect()


## Preprocesamiento independiente por fold

El imputador y el codificador se crean desde cero en cada fold y se ajustan solo con su train. Para XGBoost se conserva la escala original. En la regresión logística, `StandardScaler` se aplica dentro de la rama numérica; las categorías one-hot permanecen en 0/1.

In [ ]:
def make_preprocessor(scale_numeric: bool) -> ColumnTransformer:
    """Crea un preprocesador nuevo; el escalado queda limitado a la rama numérica."""
    numeric_steps = [
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ]
    if scale_numeric:
        # Las columnas one-hot no pasan por este escalador y conservan valores 0/1.
        numeric_steps.append(('scaler', StandardScaler()))
    return ColumnTransformer(transformers=[
        ('numeric', Pipeline(numeric_steps), NUMERIC_FEATURES),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('one_hot', OneHotEncoder(handle_unknown='ignore')),
        ]), CATEGORICAL_FEATURES),
    ])

prepared_folds = []
for raw_fold in raw_folds:
    # XGBoost conserva la escala original.
    xgb_preprocessor = make_preprocessor(scale_numeric=False)
    X_train = xgb_preprocessor.fit_transform(raw_fold['train'][MODEL_FEATURES]).astype(np.float32)
    X_validation = xgb_preprocessor.transform(raw_fold['validation'][MODEL_FEATURES]).astype(np.float32)

    # La logística usa otro preprocesador ajustado desde cero con el train del fold.
    logistic_preprocessor = make_preprocessor(scale_numeric=True)
    X_train_logistic = logistic_preprocessor.fit_transform(
        raw_fold['train'][MODEL_FEATURES]
    ).astype(np.float32)
    X_validation_logistic = logistic_preprocessor.transform(
        raw_fold['validation'][MODEL_FEATURES]
    ).astype(np.float32)
    prepared_folds.append({
        'fold': raw_fold['fold'],
        'X_train': X_train,
        'y_train': raw_fold['train'][TARGET].astype(int).reset_index(drop=True),
        'X_validation': X_validation,
        'y_validation': raw_fold['validation'][TARGET].astype(int).reset_index(drop=True),
        'X_train_logistic': X_train_logistic,
        'X_validation_logistic': X_validation_logistic,
    })
    print(raw_fold['fold'], X_train.shape, X_validation.shape)

del raw_folds
gc.collect()


## Refinamiento local de XGBoost

La región se centra en el trial 29. Se amplía `n_estimators` hacia abajo, mientras que profundidad, regularización y pesos de riesgo se buscan alrededor de los valores seleccionados. El balanceo global se fija en `False`, porque en el HPO original elevaba balanced accuracy a costa de una pérdida importante de F1.

In [ ]:
def suggest_xgboost(trial: optuna.Trial) -> dict:
    """Espacio local alrededor de la configuración XGBoost elegida en el notebook 06."""
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600, step=50),
        'max_depth': trial.suggest_int('max_depth', 7, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.08, log=True),
        'min_child_weight': trial.suggest_float('min_child_weight', 0.70, 3.00, log=True),
        'subsample': trial.suggest_float('subsample', 0.75, 0.98),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.75, 0.98),
        'gamma': trial.suggest_float('gamma', 3.0, 8.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.005, 0.30, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0, log=True),
        'weight_empty': trial.suggest_float('weight_empty', 1.50, 2.70),
        'weight_full': trial.suggest_float('weight_full', 1.50, 2.70),
    }

def evaluate_xgboost(parameters: dict, seed: int) -> pd.DataFrame:
    """Entrena un XGBoost nuevo en cada fold y registra rendimiento, iteración y tiempo."""
    rows = []
    # Los multiplicadores de clase son auxiliares y no son parámetros nativos de XGBoost.
    model_parameters = {
        key: value for key, value in parameters.items()
        if key not in {'weight_empty', 'weight_full'}
    }
    for fold in prepared_folds:
        model = XGBClassifier(
            objective='multi:softprob', num_class=3, eval_metric='mlogloss',
            tree_method='hist', early_stopping_rounds=40,
            n_jobs=-1, random_state=seed, **model_parameters,
        )
        weights = np.ones(len(fold['y_train']), dtype=np.float32)
        y_array = fold['y_train'].to_numpy()
        weights[y_array == 1] *= parameters['weight_empty']
        weights[y_array == 2] *= parameters['weight_full']

        start = time.perf_counter()
        model.fit(
            fold['X_train'], fold['y_train'], sample_weight=weights,
            eval_set=[(fold['X_validation'], fold['y_validation'])], verbose=False,
        )
        fit_seconds = time.perf_counter() - start
        prediction = model.predict(fold['X_validation']).astype(int)
        best_iteration = int(getattr(model, 'best_iteration', parameters['n_estimators'] - 1)) + 1
        rows.append({
            'model': 'xgboost_refined', 'seed': seed, 'fold': fold['fold'],
            'f1_macro': f1_score(fold['y_validation'], prediction, average='macro'),
            'balanced_accuracy': balanced_accuracy_score(fold['y_validation'], prediction),
            'best_iteration': best_iteration, 'fit_seconds': fit_seconds,
            'converged': True,
        })
        del model, prediction, weights
        gc.collect()
    return pd.DataFrame(rows)

def objective_xgboost(trial: optuna.Trial):
    parameters = suggest_xgboost(trial)
    metrics = evaluate_xgboost(parameters, RANDOM_STATE)
    trial.set_user_attr('f1_std', float(metrics['f1_macro'].std(ddof=0)))
    trial.set_user_attr('balanced_accuracy_std', float(metrics['balanced_accuracy'].std(ddof=0)))
    trial.set_user_attr('best_iterations', metrics['best_iteration'].astype(int).tolist())
    trial.set_user_attr('best_iteration_mean', float(metrics['best_iteration'].mean()))
    trial.set_user_attr('fit_seconds_mean', float(metrics['fit_seconds'].mean()))
    return float(metrics['f1_macro'].mean()), float(metrics['balanced_accuracy'].mean())

storage_url = f"sqlite:///{(OUTPUT_DIR / 'optuna_refinamiento.db').as_posix()}"
xgb_study = optuna.create_study(
    study_name='refinamiento_local_xgboost', directions=['maximize', 'maximize'],
    sampler=NSGAIISampler(seed=RANDOM_STATE), storage=storage_url, load_if_exists=True,
)
# Optuna interpreta n_trials como trials nuevos. Calculamos los pendientes para no repetir los 30 ya hechos.
completed_xgb_trials = sum(
    trial.state == optuna.trial.TrialState.COMPLETE for trial in xgb_study.trials
)
remaining_xgb_trials = max(0, N_TRIALS_XGBOOST - completed_xgb_trials)
if RUN_XGBOOST_REFINEMENT and remaining_xgb_trials:
    xgb_study.optimize(
        objective_xgboost, n_trials=remaining_xgb_trials, n_jobs=1,
        gc_after_trial=True, show_progress_bar=True,
    )
elif not RUN_XGBOOST_REFINEMENT:
    print('Refinamiento XGBoost desactivado: se reutilizan los 30 trials originales.')
else:
    print('Los 30 trials de XGBoost ya están completos; se reutilizan sin repetirlos.')
print(f'Trials XGBoost acumulados: {len(xgb_study.trials)}')


## Línea base logística L2, escalada y convergente

Se prueban únicamente cinco valores de `C` con regularización L2, solver `lbfgs`, sin pesos de clase y un máximo de 700 iteraciones. Solo se escalan las variables numéricas. Cada ajuste captura `ConvergenceWarning`; cualquier valor que no converja en los cuatro folds queda excluido.

In [ ]:
def make_logistic(C: float, seed: int) -> LogisticRegression:
    """Crea una regresión logística L2 sin balanceo automático de clases."""
    # La regularización L2 es el valor predeterminado; no pasamos `penalty` para evitar avisos deprecados.
    return LogisticRegression(
        solver='lbfgs', C=C, class_weight=None,
        max_iter=LOGISTIC_MAX_ITER, tol=LOGISTIC_TOL, random_state=seed,
    )

def evaluate_logistic(C: float, seed: int) -> pd.DataFrame:
    """Evalúa un valor fijo de C y confirma convergencia en cada fold."""
    rows = []
    for fold in prepared_folds:
        model = make_logistic(C, seed)
        start = time.perf_counter()
        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter('always', ConvergenceWarning)
            model.fit(fold['X_train_logistic'], fold['y_train'])
        fit_seconds = time.perf_counter() - start
        converged = not any(
            issubclass(warning.category, ConvergenceWarning) for warning in caught_warnings
        )
        prediction = model.predict(fold['X_validation_logistic']).astype(int)
        rows.append({
            'model': 'logistic_refined', 'C': C, 'seed': seed, 'fold': fold['fold'],
            'f1_macro': f1_score(fold['y_validation'], prediction, average='macro'),
            'balanced_accuracy': balanced_accuracy_score(fold['y_validation'], prediction),
            'best_iteration': int(model.n_iter_.max()), 'fit_seconds': fit_seconds,
            'converged': bool(converged),
        })
        del model, prediction
        gc.collect()
    return pd.DataFrame(rows)

# Esta cuadrícula sustituye el HPO de 24 trials y puede auditarse directamente.
logistic_grid_parts = []
for C in LOGISTIC_C_VALUES:
    print(f'Evaluando línea base logística con C={C}...')
    logistic_grid_parts.append(evaluate_logistic(C, RANDOM_STATE))

logistic_grid_by_fold = pd.concat(logistic_grid_parts, ignore_index=True)
logistic_grid_summary = (
    logistic_grid_by_fold.groupby(['model', 'C'], as_index=False)
    .agg(
        f1_macro_mean=('f1_macro', 'mean'),
        balanced_accuracy_mean=('balanced_accuracy', 'mean'),
        f1_macro_std=('f1_macro', lambda values: values.std(ddof=0)),
        balanced_accuracy_std=('balanced_accuracy', lambda values: values.std(ddof=0)),
        fit_seconds_mean=('fit_seconds', 'mean'),
        n_iter_max=('best_iteration', 'max'),
        converged_all_folds=('converged', 'all'),
    )
)
logistic_grid_by_fold.to_csv(
    OUTPUT_DIR / 'linea_base_logistica_por_C_y_fold.csv', index=False, encoding='utf-8-sig'
)
logistic_grid_summary.to_csv(
    OUTPUT_DIR / 'resumen_linea_base_logistica_por_C.csv', index=False, encoding='utf-8-sig'
)
display(logistic_grid_summary)


## Selección jerárquica y repetición con tres semillas

Se prioriza F1 macro. Entre configuraciones situadas como máximo a 0,003 del mejor F1, se selecciona la mayor balanced accuracy; después la menor desviación temporal y el menor tiempo. Para la logística solo se consideran valores de `C` que convergen en los cuatro folds.

In [ ]:
def trials_table(model_name: str, study: optuna.Study) -> pd.DataFrame:
    """Convierte el estudio en una tabla auditable con métricas, estabilidad y parámetros."""
    rows = []
    for trial in study.trials:
        # Los trials 30 en adelante proceden de una reanudación accidental y no forman parte del diseño.
        if model_name == 'xgboost_refined' and trial.number >= N_TRIALS_XGBOOST:
            continue
        if trial.state != optuna.trial.TrialState.COMPLETE or trial.values is None:
            continue
        row = {
            'model': model_name, 'trial_number': trial.number,
            'f1_macro_mean': trial.values[0],
            'balanced_accuracy_mean': trial.values[1],
            'f1_macro_std': trial.user_attrs.get('f1_std', np.nan),
            'balanced_accuracy_std': trial.user_attrs.get('balanced_accuracy_std', np.nan),
            'fit_seconds_mean': trial.user_attrs.get('fit_seconds_mean', np.nan),
            'best_iteration_mean': trial.user_attrs.get('best_iteration_mean', np.nan),
            'n_iter_max': trial.user_attrs.get('n_iter_max', np.nan),
            'converged_all_folds': trial.user_attrs.get('converged_all_folds', True),
        }
        row.update({f'param_{key}': value for key, value in trial.params.items()})
        rows.append(row)
    return pd.DataFrame(rows)

def select_configuration(table: pd.DataFrame, require_convergence: bool) -> pd.Series:
    candidates = table.copy()
    if require_convergence:
        candidates = candidates.loc[candidates['converged_all_folds'].eq(True)].copy()
        if candidates.empty:
            raise RuntimeError('Ningún valor de C convergió en los cuatro folds.')
    best_f1 = candidates['f1_macro_mean'].max()
    candidates = candidates.loc[candidates['f1_macro_mean'].ge(best_f1 - F1_TOLERANCE)]
    return candidates.sort_values(
        ['balanced_accuracy_mean', 'f1_macro_std', 'fit_seconds_mean', 'f1_macro_mean'],
        ascending=[False, True, True, False],
    ).iloc[0]

xgb_trials = trials_table('xgboost_refined', xgb_study)
logistic_trials = logistic_grid_summary.copy()
xgb_selected = select_configuration(xgb_trials, require_convergence=False)
logistic_selected = select_configuration(logistic_trials, require_convergence=True)

xgb_trial = next(trial for trial in xgb_study.trials if trial.number == int(xgb_selected['trial_number']))
selected_parameters = {
    'xgboost_refined': xgb_trial.params,
    'logistic_refined': {'C': float(logistic_selected['C'])},
}

xgb_trials.to_csv(OUTPUT_DIR / 'trials_refinamiento_xgboost.csv', index=False, encoding='utf-8-sig')
logistic_trials.to_csv(OUTPUT_DIR / 'configuraciones_linea_base_logistica.csv', index=False, encoding='utf-8-sig')
selection_table = pd.DataFrame([xgb_selected, logistic_selected])
selection_table.to_csv(OUTPUT_DIR / 'configuraciones_refinadas_seleccionadas.csv', index=False, encoding='utf-8-sig')
display(selection_table[[
    'model', 'f1_macro_mean', 'balanced_accuracy_mean',
    'f1_macro_std', 'fit_seconds_mean', 'n_iter_max', 'converged_all_folds',
]])


In [ ]:
# Repetimos las dos configuraciones elegidas para medir estabilidad algorítmica.
stability_parts = []
for seed in SEEDS_STABILITY:
    stability_parts.append(evaluate_xgboost(selected_parameters['xgboost_refined'], seed))
    stability_parts.append(evaluate_logistic(selected_parameters['logistic_refined']['C'], seed))

stability_by_fold = pd.concat(stability_parts, ignore_index=True)
stability_summary = (
    stability_by_fold.groupby('model', as_index=False)
    .agg(
        f1_macro_mean=('f1_macro', 'mean'),
        f1_macro_std=('f1_macro', 'std'),
        balanced_accuracy_mean=('balanced_accuracy', 'mean'),
        balanced_accuracy_std=('balanced_accuracy', 'std'),
        fit_seconds_mean=('fit_seconds', 'mean'),
        iteration_mean=('best_iteration', 'mean'),
        converged_all=('converged', 'all'),
    )
    .sort_values('f1_macro_mean', ascending=False)
)
stability_by_fold.to_csv(OUTPUT_DIR / 'estabilidad_refinamiento_por_semilla_y_fold.csv', index=False, encoding='utf-8-sig')
stability_summary.to_csv(OUTPUT_DIR / 'resumen_estabilidad_refinamiento.csv', index=False, encoding='utf-8-sig')
display(stability_summary)


## Comparación con el HPO original y exportación

La comparación es interna: usa los mismos folds de train y no representa todavía una validación externa. El notebook 07 utilizará las configuraciones ya congeladas y será el único encargado de evaluar octubre de 2022.

In [ ]:
original_summary = pd.read_csv(
    HPO_ORIGINAL_DIR / 'resumen_estabilidad_hpo.csv', encoding='utf-8-sig'
)
original_summary = original_summary.loc[original_summary['model'].isin(['xgboost', 'logistic'])].copy()
original_summary['stage'] = 'HPO original'

refined_comparison = stability_summary.rename(columns={'converged_all': 'converged'}).copy()
refined_comparison['stage'] = 'Refinamiento 06b'
comparison = pd.concat([original_summary, refined_comparison], ignore_index=True, sort=False)
comparison.to_csv(OUTPUT_DIR / 'comparacion_hpo_original_vs_refinado.csv', index=False, encoding='utf-8-sig')
display(comparison[[
    'stage', 'model', 'f1_macro_mean', 'f1_macro_std',
    'balanced_accuracy_mean', 'balanced_accuracy_std',
]])

parameter_rows = []
for model_name, parameters in selected_parameters.items():
    for parameter, value in parameters.items():
        parameter_rows.append({'model': model_name, 'parameter': parameter, 'value': value})
parameters_table = pd.DataFrame(parameter_rows)
parameters_table.to_csv(OUTPUT_DIR / 'mejores_hiperparametros_refinados.csv', index=False, encoding='utf-8-sig')
display(parameters_table)


In [ ]:
# Visualizamos el compromiso entre ambas métricas en la búsqueda local.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    xgb_trials['balanced_accuracy_mean'], xgb_trials['f1_macro_mean'],
    alpha=0.65, label='XGBoost refinado', color='#0B6E99',
)
ax.scatter(
    logistic_trials['balanced_accuracy_mean'], logistic_trials['f1_macro_mean'],
    alpha=0.65, label='Logística refinada', color='#D97904',
)
ax.set_title('Refinamiento: F1 macro frente a balanced accuracy')
ax.set_xlabel('Balanced accuracy media')
ax.set_ylabel('F1 macro medio')
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()


## Auditoría y condición de cierre

El refinamiento queda cerrado cuando existe una configuración logística convergente en todos los folds y se han repetido ambos modelos con tres semillas. Después no deben modificarse parámetros usando octubre, noviembre o diciembre. El siguiente paso es `07_modelo_final_hpo_y_validacion_externa.ipynb`.

In [ ]:
audit_table = pd.DataFrame([
    {'control': 'splits_permitidos', 'value': 'train'},
    {'control': 'ultimo_mes_leido', 'value': feature_files[-1].stem.rsplit('_', 1)[-1]},
    {'control': 'test_consultado', 'value': False},
    {'control': 'n_folds_temporales', 'value': len(FOLD_SPECS)},
    {'control': 'n_semillas_estabilidad', 'value': len(SEEDS_STABILITY)},
    {'control': 'logistica_convergente', 'value': bool(stability_summary.loc[
        stability_summary['model'].eq('logistic_refined'), 'converged_all'
    ].iloc[0])},
])
audit_table.to_csv(OUTPUT_DIR / 'auditoria_refinamiento_sin_fuga.csv', index=False, encoding='utf-8-sig')
display(audit_table)

assert feature_files[-1].stem.endswith('202209')
assert not bool(audit_table.loc[audit_table['control'].eq('test_consultado'), 'value'].iloc[0])
assert bool(audit_table.loc[audit_table['control'].eq('logistica_convergente'), 'value'].iloc[0])
print('Refinamiento cerrado sin consultar octubre–diciembre. Ya puede prepararse el notebook 07.')
